# T33 - 信用债规模 - 重构 Notebook

## 项目概述

信用债规模项目专注于统计和分析债券市场的存续规模数据，包括不同类型（信用债、金融债、ABS等）、不同期限、不同评级、不同发行人类型的债券余额分布。

### 功能描述
- 债券分类重建（信用债、金融债、ABS）
- 评级补全
- 久期补全
- 多维度汇总统计

### 数据源
- MySQL数据库：bond库（basicinfo_credit, basicinfo_finance, basicinfo_abs, marketinfo_*）
- PostgreSQL数据库：tsdb库（债券新分类表）

### 输出结果
- bond.债券分类表：债券分类汇总表
- bond.marketinfo3：市场信息汇总表
- bond.信用债规模：规模汇总表

---

## 1. 环境配置

导入依赖和配置参数

In [ ]:
# 导入依赖
import os
import sys
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text
from datetime import datetime, date, timedelta
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# 导入配置
from config import (
    PROJECT_ROOT, DATA_DIR, OUTPUT_DIR,
    get_mysql_engine, get_postgres_engine,
    MAJOR_TYPES, RATING_LEVELS, DURATION_THRESHOLDS
)

print("环境配置完成")
print(f"项目根目录: {PROJECT_ROOT}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")

---

## 2. 数据获取

建立数据库连接并获取数据

In [ ]:
# 建立数据库连接
mysql_engine = get_mysql_engine()
mysql_conn = mysql_engine.connect()

postgres_engine = get_postgres_engine()

print("数据库连接成功")
print(f"MySQL: {mysql_engine.url}")
print(f"PostgreSQL: {postgres_engine.url}")

In [ ]:
# 测试数据获取 - 查看基础信息表结构
def get_table_sample(table_name, limit=5):
    """获取表样本数据"""
    sql = f"SELECT * FROM {table_name} LIMIT {limit}"
    df = pd.read_sql(text(sql), mysql_conn)
    return df

# 查看信用债基础信息样本
credit_sample = get_table_sample('bond.basicinfo_credit')
print("信用债基础信息样本:")
print(credit_sample.head())

---

## 3. 数据处理

债券分类、评级补全、久期补全

### 3.1 债券分类重建

In [ ]:
def rebuild_bond_classification():
    """
    重建债券分类表
    
    处理逻辑:
    1. 清空债券分类表
    2. 从信用债基础表提取并分类（城投/产业）
    3. 从金融债基础表提取金融类
    4. 从PostgreSQL获取ABS分类
    """
    
    # Step 1: 清空债券分类表
    sql_truncate = "TRUNCATE bond.债券分类"
    mysql_conn.execute(text(sql_truncate))
    mysql_conn.commit()
    print("[Step 1] 债券分类表已清空")
    
    # Step 2: 插入信用债分类（城投/产业）
    sql_credit = """
    INSERT IGNORE INTO bond.债券分类 (trade_code, 大类, 子类)
    SELECT
        trade_code,
        CASE
            WHEN ths_is_city_invest_debt_yy_bond = '否' THEN '产业'
            ELSE '城投'
        END AS 大类,
        CASE
            WHEN ths_is_city_invest_debt_yy_bond = '否' THEN 
                CASE
                    WHEN ths_object_the_sw_bond LIKE '%%银行%%' THEN ''
                    WHEN ths_object_the_sw_bond LIKE '%%证券%%' THEN '非银金融--多元金融--其他多元金融'
                    WHEN ths_object_the_sw_bond LIKE '%%保险%%' THEN '非银金融--多元金融--其他多元金融'
                    ELSE ths_object_the_sw_bond
                END
            ELSE ths_urban_platform_area_bond
        END AS 子类
    FROM bond.basicinfo_credit
    WHERE ths_object_the_sw_bond NOT IN (
        '银行--其他银行Ⅱ--其他银行Ⅲ',
        '银行--银行--银行',
        '银行--城商行Ⅱ--城商行Ⅲ'
    )
    """
    mysql_conn.execute(text(sql_credit))
    mysql_conn.commit()
    print("[Step 2] 信用债分类（城投/产业）已插入")
    
    # Step 3: 插入银行类债券到金融分类
    sql_bank = """
    INSERT IGNORE INTO bond.债券分类 (trade_code, 大类, 子类)
    SELECT
        trade_code,
        '金融' AS 大类,
        '银行' AS 子类
    FROM bond.basicinfo_credit
    WHERE ths_object_the_sw_bond IN (
        '银行--其他银行Ⅱ--其他银行Ⅲ',
        '银行--银行--银行',
        '银行--城商行Ⅱ--城商行Ⅲ'
    )
    """
    mysql_conn.execute(text(sql_bank))
    mysql_conn.commit()
    print("[Step 3] 银行类债券已插入到金融分类")
    
    # Step 4: 插入金融债分类
    sql_finance = """
    INSERT IGNORE INTO bond.债券分类 (trade_code, 大类, 子类)
    SELECT
        trade_code,
        '金融' AS 大类,
        CASE
            WHEN ths_org_type_bond LIKE '%%银行%%' THEN '银行'
            WHEN ths_org_type_bond LIKE '%%证券公司%%' THEN '证券公司'
            WHEN ths_org_type_bond LIKE '%%保险%%' THEN '保险'
            WHEN ths_ths_bond_third_type_bond = '其他非银金融机构债' THEN '其他'
        END AS 子类
    FROM bond.basicinfo_finance
    """
    mysql_conn.execute(text(sql_finance))
    mysql_conn.commit()
    print("[Step 4] 金融债分类已插入")
    
    # Step 5: 插入ABS分类（从PostgreSQL获取）
    sql_abs = """
    SELECT
        trade_code,
        'ABS' AS 大类,
        abs类型 AS 子类
    FROM 债券新分类
    WHERE abs类型 IS NOT NULL AND abs类型 != ''
    """
    df_abs = pd.read_sql(sql_abs, postgres_engine)
    df_abs.to_sql('债券分类', mysql_conn, if_exists='append', index=False)
    mysql_conn.commit()
    print(f"[Step 5] ABS分类已插入 ({len(df_abs)} 条)")
    
    print("\n债券分类重建完成!")

# 执行债券分类重建
rebuild_bond_classification()

### 3.2 评级补全

In [ ]:
def update_ratings():
    """
    补全债券评级信息
    
    处理逻辑:
    1. 从marketinfo表获取每个债券最新日期的评级
    2. 更新到债券分类表
    """
    
    table_names = [
        'bond.marketinfo_credit',
        'bond.marketinfo_finance',
        'bond.marketinfo_abs'
    ]
    
    for table_name in table_names:
        sql = f"""
        UPDATE
            bond.`债券分类`
        INNER JOIN (
            SELECT
                mc.trade_code,
                mc.ths_cb_market_implicit_rating_bond
            FROM
                {table_name} mc
            JOIN (
                SELECT
                    trade_code,
                    MAX(dt) AS max_dt
                FROM
                    {table_name}
                WHERE
                    ths_cb_market_implicit_rating_bond IS NOT NULL
                    AND ths_cb_market_implicit_rating_bond != '0'
                GROUP BY
                    trade_code
            ) subq
            ON
                mc.trade_code = subq.trade_code
                AND mc.dt = subq.max_dt
        ) AS derived_table
        ON
            bond.`债券分类`.trade_code = derived_table.trade_code
        SET
            bond.`债券分类`.评级 = derived_table.ths_cb_market_implicit_rating_bond
        """
        mysql_conn.execute(text(sql))
        mysql_conn.commit()
        print(f"评级更新完成: {table_name}")
    
    print("\n评级补全完成!")

# 执行评级补全
update_ratings()

### 3.3 久期补全

In [ ]:
def update_duration():
    """
    补全久期信息
    
    处理逻辑:
    1. 从市场表提取余额和修正久期
    2. 久期缺失时从基础信息表计算
    3. 永续债特殊处理
    """
    
    # 定义数据源映射
    source_tables = [
        ('bond.marketinfo_credit', 'basicinfo_credit'),
        ('bond.marketinfo_finance', 'basicinfo_finance'),
        ('bond.marketinfo_abs', 'basicinfo_abs')
    ]
    
    for market_table, basic_table in source_tables:
        print(f"\n处理: {market_table}")
        
        # Step 1: 插入市场数据
        sql_insert = f"""
        INSERT INTO bond.marketinfo3 (DT, trade_code, ths_bond_balance_bond, 久期)
        SELECT
            DT,
            trade_code,
            ths_bond_balance_bond,
            CASE
                WHEN ths_evaluate_modified_dur_cb_bond_exercise > 0
                THEN ths_evaluate_modified_dur_cb_bond_exercise
                ELSE ths_evaluate_interest_durcb_bond_exercise + ths_evaluate_interest_durcb_bond_exercise
            END AS 久期
        FROM {market_table}
        WHERE ths_bond_balance_bond > 0
        ON DUPLICATE KEY UPDATE
            ths_bond_balance_bond = VALUES(ths_bond_balance_bond),
            久期 = VALUES(久期)
        """
        mysql_conn.execute(text(sql_insert))
        mysql_conn.commit()
        print(f"  [Step 1] 市场数据已插入")
        
        # Step 2: 创建临时表计算缺失久期
        sql_temp = "DROP TEMPORARY TABLE IF EXISTS temp_table"
        mysql_conn.execute(text(sql_temp))
        mysql_conn.commit()
        
        sql_create_temp = f"""
        CREATE TEMPORARY TABLE temp_table AS
        SELECT
            A.dt,
            B.trade_code,
            CASE
                WHEN B.ths_bond_maturity_theory_bond LIKE '%(%)%'
                    AND SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) REGEXP '^[0-9]+$'
                    AND DATEDIFF(
                        DATE_ADD(
                            B.ths_interest_begin_date_bond,
                            INTERVAL CAST(SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) AS UNSIGNED) YEAR
                        ),
                        A.DT
                    ) / 365 >= 0
                THEN
                    DATEDIFF(
                        DATE_ADD(
                            B.ths_interest_begin_date_bond,
                            INTERVAL CAST(SUBSTRING_INDEX(SUBSTRING_INDEX(B.ths_bond_maturity_theory_bond, '(', -1), '+', 1) AS UNSIGNED) YEAR
                        ),
                        A.DT
                    ) / 365
                ELSE
                    DATEDIFF(B.ths_maturity_date_bond, A.DT) / 365
            END AS new_duration
        FROM marketinfo3 A
        JOIN {basic_table} B ON A.trade_code = B.trade_code
        WHERE A.久期 IS NULL
        """
        mysql_conn.execute(text(sql_create_temp))
        mysql_conn.commit()
        print(f"  [Step 2] 临时表已创建")
        
        # Step 3: 更新缺失久期
        sql_update = """
        UPDATE marketinfo3 A
        JOIN temp_table T
            ON A.trade_code = T.trade_code
            AND A.dt = T.dt
        SET A.久期 = T.new_duration
        WHERE A.久期 IS NULL
        """
        mysql_conn.execute(text(sql_update))
        mysql_conn.commit()
        print(f"  [Step 3] 缺失久期已更新")
    
    print("\n久期补全完成!")

# 执行久期补全
update_duration()

---

## 4. 核心逻辑

多维度规模汇总

In [ ]:
def calculate_scale_summary():
    """
    计算多维度规模汇总
    
    汇总维度:
    - 大类: 全部、城投、产业、金融、ABS
    - 省市: 城投按省、市分类
    - 行业: 产业按申万行业分类
    - 金融机构: 银行、证券公司、保险、其他
    - ABS: 按资产类型
    - 评级: AAA、AA+、AA等
    - 久期: 0.5年、1年、2年、3年、4年、5年、7年、10年
    """
    
    # 定义所有汇总查询
    queries = [
        # 1. 全部汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            DT,
            '全部' AS 分类,
            '' AS 子类,
            '大类' AS 分类方式,
            SUM(ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3
        WHERE 久期 IS NOT NULL
        GROUP BY DT
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 2. 按大类汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            B.大类 AS 分类,
            '' AS 子类,
            '大类' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        WHERE A.久期 IS NOT NULL
        GROUP BY A.DT, B.大类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 3. 城投按省汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            '城投' AS 分类,
            C.省 AS 子类,
            '省' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_xzqh_ct C ON B.子类 = C.`城投区域`
        WHERE B.大类 = '城投'
            AND C.`省` IS NOT NULL
            AND C.`省` != ''
            AND A.久期 IS NOT NULL
        GROUP BY A.DT, C.省
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 4. 城投按市汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            '城投' AS 分类,
            C.市 AS 子类,
            '市' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_xzqh_ct C ON B.子类 = C.`城投区域`
        WHERE B.大类 = '城投'
            AND C.`市` IS NOT NULL
            AND C.`市` != ''
            AND A.久期 IS NOT NULL
        GROUP BY A.DT, C.市
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 5. 产业按申万一级行业汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            '产业' AS 分类,
            C.申万一级 AS 子类,
            '申万一级' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_industrytype1 C ON B.子类 = C.申万行业
        WHERE B.大类 = '产业'
            AND C.`申万一级` IS NOT NULL
            AND C.`申万一级` != ''
            AND A.久期 IS NOT NULL
        GROUP BY A.DT, C.申万一级
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 6. 产业按申万行业汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            '产业' AS 分类,
            B.子类,
            '申万行业' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        WHERE B.大类 = '产业'
            AND A.久期 IS NOT NULL
        GROUP BY A.DT, B.子类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 7. 产业按一级分类汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            '产业' AS 分类,
            C.一级分类 AS 子类,
            '一级分类' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_industrytype1 C ON B.子类 = C.申万行业
        WHERE B.大类 = '产业'
            AND C.`一级分类` IS NOT NULL
            AND C.`一级分类` != ''
            AND A.久期 IS NOT NULL
        GROUP BY A.DT, C.一级分类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 8. 产业按二级分类汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            '产业' AS 分类,
            C.二级分类 AS 子类,
            '二级分类' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        INNER JOIN bond.basicinfo_industrytype1 C ON B.子类 = C.申万行业
        WHERE B.大类 = '产业'
            AND C.`二级分类` IS NOT NULL
            AND C.`二级分类` != ''
            AND A.久期 IS NOT NULL
        GROUP BY A.DT, C.二级分类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 9. 金融按机构类型汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            '金融' AS 分类,
            B.子类 AS 子类,
            '金融机构' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        WHERE B.`大类` = '金融'
            AND B.子类 IS NOT NULL
            AND B.子类 != ''
            AND A.久期 IS NOT NULL
        GROUP BY A.DT, B.子类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 10. ABS按资产类型汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            'ABS' AS 分类,
            B.子类 AS 子类,
            '资产' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        WHERE B.大类 = 'ABS'
            AND B.子类 IS NOT NULL
            AND B.子类 != ''
            AND A.久期 IS NOT NULL
        GROUP BY A.DT, B.子类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 11. 全部按评级汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            '全部' AS 分类,
            B.评级 AS 子类,
            '评级' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        WHERE A.久期 IS NOT NULL
        GROUP BY A.DT, B.评级
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 12. 按大类+评级汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            B.`大类` AS 分类,
            B.评级 AS 子类,
            '评级' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        WHERE A.久期 IS NOT NULL
        GROUP BY A.DT, B.`大类`, B.评级
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 13. 全部按久期汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            DT,
            '全部' AS 分类,
            CASE
                WHEN 久期 < 0.75 THEN 0.5
                WHEN 久期 < 1.5 THEN 1
                WHEN 久期 < 2.5 THEN 2
                WHEN 久期 < 3.5 THEN 3
                WHEN 久期 < 4.5 THEN 4
                WHEN 久期 < 6 THEN 5
                WHEN 久期 < 8 THEN 7
                ELSE 10
            END AS 子类,
            '久期' AS 分类方式,
            SUM(ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3
        WHERE 久期 IS NOT NULL
        GROUP BY DT, 子类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """,
        
        # 14. 按大类+久期汇总
        """
        INSERT INTO bond.信用债规模 (DT, 分类, 子类, 分类方式, 余额)
        SELECT
            A.DT,
            B.`大类` AS 分类,
            CASE
                WHEN 久期 < 0.75 THEN 0.5
                WHEN 久期 < 1.5 THEN 1
                WHEN 久期 < 2.5 THEN 2
                WHEN 久期 < 3.5 THEN 3
                WHEN 久期 < 4.5 THEN 4
                WHEN 久期 < 6 THEN 5
                WHEN 久期 < 8 THEN 7
                ELSE 10
            END AS 子类,
            '久期' AS 分类方式,
            SUM(A.ths_bond_balance_bond) AS 余额
        FROM bond.marketinfo3 A
        INNER JOIN bond.债券分类 B ON A.trade_code = B.trade_code
        WHERE 久期 IS NOT NULL
        GROUP BY A.DT, B.`大类`, 子类
        ON DUPLICATE KEY UPDATE 余额 = VALUES(余额)
        """
    ]
    
    # 执行所有查询
    for i, query in enumerate(queries, 1):
        try:
            mysql_conn.execute(text(query))
            mysql_conn.commit()
            print(f"查询 {i}/{len(queries)} 执行成功")
        except Exception as e:
            print(f"查询 {i}/{len(queries)} 执行失败: {str(e)}")
    
    print("\n规模汇总计算完成!")

# 执行规模汇总
calculate_scale_summary()

---

## 5. 执行与测试

主流程执行和结果验证

In [ ]:
def run_full_etl():
    """
    执行完整的ETL流程
    
    流程:
    1. 债券分类重建
    2. 评级补全
    3. 久期补全
    4. 规模汇总计算
    """
    print("="*60)
    print("开始执行信用债规模ETL流程")
    print("="*60)
    
    start_time = datetime.now()
    
    try:
        # Step 1: 债券分类重建
        print("\n[Step 1/4] 债券分类重建...")
        rebuild_bond_classification()
        
        # Step 2: 评级补全
        print("\n[Step 2/4] 评级补全...")
        update_ratings()
        
        # Step 3: 久期补全
        print("\n[Step 3/4] 久期补全...")
        update_duration()
        
        # Step 4: 规模汇总
        print("\n[Step 4/4] 规模汇总计算...")
        calculate_scale_summary()
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        
        print("\n" + "="*60)
        print(f"ETL流程执行完成! 耗时: {duration:.2f} 秒")
        print("="*60)
        
    except Exception as e:
        print(f"\nETL流程执行失败: {str(e)}")
        raise

# 执行完整ETL
run_full_etl()

In [ ]:
def verify_results():
    """
    验证ETL结果
    """
    print("\n结果验证:")
    print("-"*40)
    
    # 验证债券分类表
    sql_classification = "SELECT 大类, COUNT(*) as cnt FROM bond.债券分类 GROUP BY 大类"
    df_classification = pd.read_sql(text(sql_classification), mysql_conn)
    print("\n债券分类统计:")
    print(df_classification)
    
    # 验证市场信息3表
    sql_marketinfo3 = """
    SELECT
        COUNT(*) as total_records,
        COUNT(DISTINCT trade_code) as bond_count,
        COUNT(DISTINCT DT) as date_count,
        SUM(CASE WHEN 久期 IS NOT NULL THEN 1 ELSE 0 END) as duration_filled
    FROM bond.marketinfo3
    """
    df_marketinfo3 = pd.read_sql(text(sql_marketinfo3), mysql_conn)
    print("\n市场信息3表统计:")
    print(df_marketinfo3)
    
    # 验证规模汇总表
    sql_scale = """
    SELECT
        分类方式,
        COUNT(DISTINCT 分类) as category_count,
        COUNT(*) as record_count
    FROM bond.信用债规模
    GROUP BY 分类方式
    ORDER BY record_count DESC
    """
    df_scale = pd.read_sql(text(sql_scale), mysql_conn)
    print("\n规模汇总统计:")
    print(df_scale)

# 执行验证
verify_results()

---

## 6. 工具函数

可复用的辅助函数

In [ ]:
def classify_term(term):
    """
    期限分类
    
    Args:
        term: 期限年数
    
    Returns:
        tuple: (期限类别, 代表值)
    """
    if pd.isna(term):
        return '未知', None
    
    if term < 0.75:
        return '短期', 0.5
    elif term < 1.5:
        return '中短期', 1
    elif term < 2.5:
        return '中短期', 2
    elif term < 3.5:
        return '中期', 3
    elif term < 4.5:
        return '中期', 4
    elif term < 6:
        return '中长期', 5
    elif term < 8:
        return '长期', 7
    else:
        return '超长期', 10


def standardize_rating(rating):
    """
    评级标准化
    
    Args:
        rating: 原始评级字符串
    
    Returns:
        str: 标准化评级
    """
    if pd.isna(rating) or rating == '' or rating == '--' or rating == '0':
        return '无评级'
    
    rating_str = str(rating)
    
    if 'AAA' in rating_str:
        return 'AAA'
    elif 'AA+' in rating_str:
        return 'AA+'
    elif 'AA-' in rating_str:
        return 'AA-'
    elif 'AA' in rating_str:
        return 'AA'
    elif 'A' in rating_str:
        return 'A'
    else:
        return '其他'


def calculate_hhi(series):
    """
    计算HHI指数（赫芬达尔-赫希曼指数）
    
    Args:
        series: 规模数据Series
    
    Returns:
        float: HHI指数
    """
    total = series.sum()
    if total == 0:
        return 0
    shares = series / total
    return (shares ** 2).sum()


def calculate_cr(series, n=10):
    """
    计算CRn（前n大占比）
    
    Args:
        series: 规模数据Series
        n: 前n大
    
    Returns:
        float: CRn占比
    """
    total = series.sum()
    if total == 0:
        return 0
    top_n = series.sort_values(ascending=False).head(n).sum()
    return top_n / total


print("工具函数定义完成")

In [ ]:
# 关闭数据库连接
mysql_conn.close()
print("数据库连接已关闭")